In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/dataset_dict.json
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/validation/state.json
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/validation/dataset_info.json
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/validation/data-00000-of-00001.arrow
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/test/state.json
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/test/dataset_info.json
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/test/data-00000-of-00001.arrow
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/train/data-00001-of-00003.arrow
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/train/state.json
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/train/dataset_info.json
/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC/train/data-00002-of-00003.arrow
/kaggle/input/datasets/laikasarfraz/asr-arabic/F

In [2]:
# ============================================================
# PART 1 : INSTALL LIBRARIES & SETUP
# Arabic Automatic Speech Recognition (ASR)
# FLEURS Arabic Dataset
# Kaggle 2026 Compatible
# ============================================================

print("=" * 70)
print("INSTALLING REQUIRED LIBRARIES")
print("=" * 70)

# Upgrade pip
!pip -q install --upgrade pip

# Install latest stable libraries
!pip -q install -U \
transformers \
datasets[audio] \
accelerate \
evaluate \
jiwer \
librosa \
soundfile \
sentencepiece

print("\n✓ Libraries Installed Successfully")

# ============================================================
# IMPORT BASIC LIBRARIES
# ============================================================

import os
import random
import warnings
import numpy as np
import torch
import transformers
import datasets
import evaluate

warnings.filterwarnings("ignore")

# ============================================================
# RANDOM SEED
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# ENVIRONMENT INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("ENVIRONMENT INFORMATION")
print("=" * 70)

print(f"Python Version       : {os.sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"Transformers Version : {transformers.__version__}")
print(f"Datasets Version     : {datasets.__version__}")
print(f"Evaluate Version     : {evaluate.__version__}")

print(f"\nDevice               : {device}")

if torch.cuda.is_available():
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version         : {torch.version.cuda}")

print("\n✓ Environment Ready")

INSTALLING REQUIRED LIBRARIES
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.5 MB/s eta 0:00:0000:0100:01

✓ Libraries Installed Successfully

ENVIRONMENT INFORMATION
Python Version       : 3.12.13
PyTorch Version      : 2.10.0+cu128
Transformers Version : 5.13.1
Datasets Version     : 5.0.0
Evaluate Version     : 0.4.6

Device               : cuda
GPU                  : Tesla T4
CUDA Version         : 12.8

✓ Environment Ready


In [3]:
# ============================================================
# PART 2 : IMPORT LIBRARIES & CONFIGURATION
# Arabic ASR - FLEURS
# ============================================================

print("=" * 70)
print("IMPORTING LIBRARIES")
print("=" * 70)

# ============================================================
# STANDARD LIBRARIES
# ============================================================

import re
import json
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Union

# ============================================================
# DATASETS
# ============================================================

from datasets import (
    load_dataset,
    Audio
)

# ============================================================
# TRANSFORMERS
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
    Trainer,
    TrainingArguments
)

# ============================================================
# EVALUATION
# ============================================================

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# ============================================================
# CONFIGURATION
# ============================================================

LANGUAGE = "ar_eg"

MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

SAMPLING_RATE = 16000

BATCH_SIZE = 8

GRADIENT_ACCUMULATION_STEPS = 2

LEARNING_RATE = 1e-4

NUM_EPOCHS = 20

WARMUP_STEPS = 500

WEIGHT_DECAY = 0.005

OUTPUT_DIR = "./wav2vec2-arabic-fleurs"

# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(f"Language                     : {LANGUAGE}")
print(f"Model                        : {MODEL_NAME}")
print(f"Sampling Rate                : {SAMPLING_RATE}")
print(f"Batch Size                   : {BATCH_SIZE}")
print(f"Gradient Accumulation Steps  : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning Rate                : {LEARNING_RATE}")
print(f"Epochs                       : {NUM_EPOCHS}")
print(f"Warmup Steps                 : {WARMUP_STEPS}")
print(f"Weight Decay                 : {WEIGHT_DECAY}")
print(f"Output Directory             : {OUTPUT_DIR}")

print("\n✓ Configuration Loaded Successfully")

IMPORTING LIBRARIES



TRAINING CONFIGURATION
Language                     : ar_eg
Model                        : jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Sampling Rate                : 16000
Batch Size                   : 8
Gradient Accumulation Steps  : 2
Learning Rate                : 0.0001
Epochs                       : 20
Warmup Steps                 : 500
Weight Decay                 : 0.005
Output Directory             : ./wav2vec2-arabic-fleurs

✓ Configuration Loaded Successfully


In [4]:
# ============================================================
# PART 2 : IMPORT LIBRARIES & CONFIGURATION
# Arabic ASR - FLEURS
# ============================================================

print("=" * 70)
print("IMPORTING LIBRARIES")
print("=" * 70)

# ============================================================
# STANDARD LIBRARIES
# ============================================================

import re
import json
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Union

# ============================================================
# DATASETS
# ============================================================

from datasets import (
    load_dataset,
    Audio
)

# ============================================================
# TRANSFORMERS
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
    Trainer,
    TrainingArguments
)

# ============================================================
# EVALUATION
# ============================================================

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# ============================================================
# CONFIGURATION
# ============================================================

LANGUAGE = "ar_eg"

MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

SAMPLING_RATE = 16000

BATCH_SIZE = 8

GRADIENT_ACCUMULATION_STEPS = 2

LEARNING_RATE = 1e-4

NUM_EPOCHS = 20

WARMUP_STEPS = 500

WEIGHT_DECAY = 0.005

OUTPUT_DIR = "./wav2vec2-arabic-fleurs"

# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(f"Language                     : {LANGUAGE}")
print(f"Model                        : {MODEL_NAME}")
print(f"Sampling Rate                : {SAMPLING_RATE}")
print(f"Batch Size                   : {BATCH_SIZE}")
print(f"Gradient Accumulation Steps  : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning Rate                : {LEARNING_RATE}")
print(f"Epochs                       : {NUM_EPOCHS}")
print(f"Warmup Steps                 : {WARMUP_STEPS}")
print(f"Weight Decay                 : {WEIGHT_DECAY}")
print(f"Output Directory             : {OUTPUT_DIR}")

print("\n✓ Configuration Loaded Successfully")

IMPORTING LIBRARIES

TRAINING CONFIGURATION
Language                     : ar_eg
Model                        : jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Sampling Rate                : 16000
Batch Size                   : 8
Gradient Accumulation Steps  : 2
Learning Rate                : 0.0001
Epochs                       : 20
Warmup Steps                 : 500
Weight Decay                 : 0.005
Output Directory             : ./wav2vec2-arabic-fleurs

✓ Configuration Loaded Successfully


In [7]:
# ============================================================
# PART 3 : LOAD FLEURS ARABIC DATASET (LOCAL KAGGLE DATASET)
# ============================================================

print("=" * 70)
print("LOADING FLEURS ARABIC DATASET")
print("=" * 70)

from datasets import load_from_disk, Audio

# Change this path according to your Kaggle dataset name
DATASET_PATH = "/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC"

dataset = load_from_disk(DATASET_PATH)

train_dataset = dataset["train"]
validation_dataset = dataset["validation"]
test_dataset = dataset["test"]

# Convert audio to 16 kHz
train_dataset = train_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

validation_dataset = validation_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

test_dataset = test_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

print("\nDataset Loaded Successfully")
print("-" * 70)

print("Training Samples   :", len(train_dataset))
print("Validation Samples :", len(validation_dataset))
print("Testing Samples    :", len(test_dataset))

print("\nDataset Features")
print(train_dataset.features)

sample = train_dataset[0]

print("\nSample Transcript")
print("-" * 70)
print(sample["transcription"])

LOADING FLEURS ARABIC DATASET

Dataset Loaded Successfully
----------------------------------------------------------------------
Training Samples   : 2104
Validation Samples : 295
Testing Samples    : 428

Dataset Features
{'id': Value('int32'), 'num_samples': Value('int32'), 'path': Value('string'), 'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'transcription': Value('string'), 'raw_transcription': Value('string'), 'gender': ClassLabel(names=['male', 'female', 'other']), 'lang_id': ClassLabel(names=['af_za', 'am_et', 'ar_eg', 'as_in', 'ast_es', 'az_az', 'be_by', 'bg_bg', 'bn_in', 'bs_ba', 'ca_es', 'ceb_ph', 'ckb_iq', 'cmn_hans_cn', 'cs_cz', 'cy_gb', 'da_dk', 'de_de', 'el_gr', 'en_us', 'es_419', 'et_ee', 'fa_ir', 'ff_sn', 'fi_fi', 'fil_ph', 'fr_fr', 'ga_ie', 'gl_es', 'gu_in', 'ha_ng', 'he_il', 'hi_in', 'hr_hr', 'hu_hu', 'hy_am', 'id_id', 'ig_ng', 'is_is', 'it_it', 'ja_jp', 'jv_id', 'ka_ge', 'kam_ke', 'kea_cv', 'kk_kz', 'km_kh', 'kn_in', 'ko_kr

In [8]:
# ============================================================
# PART 4 : LOAD PROCESSOR & MODEL
# Arabic ASR - FLEURS
# ============================================================

print("=" * 70)
print("LOADING PROCESSOR & MODEL")
print("=" * 70)

from transformers import (
    AutoProcessor,
    AutoModelForCTC
)

# ------------------------------------------------------------
# LOAD PROCESSOR
# ------------------------------------------------------------

print("Loading Processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

print("✓ Processor Loaded")

# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

print("\nLoading Model...")

model = AutoModelForCTC.from_pretrained(
    MODEL_NAME
)

print("✓ Model Loaded")

# ------------------------------------------------------------
# FREEZE FEATURE ENCODER
# ------------------------------------------------------------

if hasattr(model, "freeze_feature_encoder"):
    model.freeze_feature_encoder()
    print("✓ Feature Encoder Frozen")

# ------------------------------------------------------------
# MOVE MODEL TO DEVICE
# ------------------------------------------------------------

model.to(device)

print(f"✓ Model moved to {device}")

# ------------------------------------------------------------
# MODEL INFORMATION
# ------------------------------------------------------------

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("\n" + "=" * 70)
print("MODEL SUMMARY")
print("=" * 70)

print(f"Model                : {MODEL_NAME}")
print(f"Vocabulary Size      : {model.config.vocab_size}")
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Frozen Parameters    : {frozen_params:,}")
print(f"CTC Loss Reduction   : {model.config.ctc_loss_reduction}")
print(f"Pad Token ID         : {model.config.pad_token_id}")

print("\n✓ Processor and Model Ready")

LOADING PROCESSOR & MODEL
Loading Processor...


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

✓ Processor Loaded

Loading Model...


pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

✓ Model Loaded
✓ Feature Encoder Frozen
✓ Model moved to cuda

MODEL SUMMARY
Model                : jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Vocabulary Size      : 51
Total Parameters     : 315,490,995
Trainable Parameters : 311,280,819
Frozen Parameters    : 4,210,176
CTC Loss Reduction   : mean
Pad Token ID         : 0

✓ Processor and Model Ready


In [10]:
from datasets import load_from_disk
import shutil
import os

# Read-only dataset
SOURCE = "/kaggle/input/datasets/laikasarfraz/asr-arabic/FLEURS_ARABIC"

# Writable location
DEST = "/kaggle/working/FLEURS_ARABIC"

# Copy only once
if not os.path.exists(DEST):
    shutil.copytree(SOURCE, DEST)

dataset = load_from_disk(DEST)

train_dataset = dataset["train"]
validation_dataset = dataset["validation"]
test_dataset = dataset["test"]

print("Dataset copied to writable directory.")

Dataset copied to writable directory.


In [11]:
# ============================================================
# PART 5 : DATA PREPROCESSING
# Arabic ASR - FLEURS
# Compatible with AutoProcessor
# ============================================================

print("=" * 70)
print("PREPROCESSING DATASET")
print("=" * 70)

import numpy as np

bad_samples = []

# ------------------------------------------------------------
# PREPROCESS FUNCTION
# ------------------------------------------------------------

def prepare_dataset(batch, idx):

    try:

        # Decode Audio
        audio = batch["audio"]

        audio_samples = audio.get_all_samples()

        speech = audio_samples.data.numpy().squeeze().astype(np.float32)

        sampling_rate = audio.metadata.sample_rate

        # Input Values
        batch["input_values"] = processor(
            speech,
            sampling_rate=sampling_rate
        ).input_values[0]

        # Input Length
        batch["input_length"] = len(batch["input_values"])

        # Labels
        batch["labels"] = processor(
            text=batch["transcription"]
        ).input_ids

        return batch

    except Exception as e:

        bad_samples.append(
            {
                "index": idx,
                "error": str(e)
            }
        )

        return None


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

print("Processing Training Dataset...")

train_dataset = train_dataset.map(
    prepare_dataset,
    with_indices=True,
)

train_dataset = train_dataset.filter(
    lambda x: x is not None
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

print("Processing Validation Dataset...")

validation_dataset = validation_dataset.map(
    prepare_dataset,
    with_indices=True,
)

validation_dataset = validation_dataset.filter(
    lambda x: x is not None
)

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

print("Processing Test Dataset...")

test_dataset = test_dataset.map(
    prepare_dataset,
    with_indices=True,
)

test_dataset = test_dataset.filter(
    lambda x: x is not None
)

# ------------------------------------------------------------
# KEEP REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "input_values",
    "input_length",
    "labels"
]

train_dataset = train_dataset.remove_columns(
    [
        c for c in train_dataset.column_names
        if c not in required_columns
    ]
)

validation_dataset = validation_dataset.remove_columns(
    [
        c for c in validation_dataset.column_names
        if c not in required_columns
    ]
)

test_dataset = test_dataset.remove_columns(
    [
        c for c in test_dataset.column_names
        if c not in required_columns
    ]
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREPROCESSING SUMMARY")
print("=" * 70)

print("Training Samples   :", len(train_dataset))
print("Validation Samples :", len(validation_dataset))
print("Testing Samples    :", len(test_dataset))

print("Bad Samples        :", len(bad_samples))

if len(bad_samples) > 0:
    print("\nExample Error:")
    print(bad_samples[0])

print("\n✓ Dataset preprocessing completed successfully.")

PREPROCESSING DATASET
Processing Training Dataset...


Map:   0%|          | 0/2104 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2103 [00:00<?, ? examples/s]

Processing Validation Dataset...


Map:   0%|          | 0/295 [00:00<?, ? examples/s]

Filter:   0%|          | 0/295 [00:00<?, ? examples/s]

Processing Test Dataset...


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

Filter:   0%|          | 0/427 [00:00<?, ? examples/s]


PREPROCESSING SUMMARY
Training Samples   : 2103
Validation Samples : 295
Testing Samples    : 427
Bad Samples        : 2

Example Error:
{'index': 1652, 'error': 'Could not receive frame from decoder: Invalid data found when processing input'}

✓ Dataset preprocessing completed successfully.


In [12]:
# ============================================================
# PART 6 : DATA COLLATOR
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List, Union

import torch

print("=" * 70)
print("CREATING DATA COLLATOR")
print("=" * 70)


@dataclass
class DataCollatorCTCWithPadding:

    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features):

        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features=input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("✓ Data Collator Created Successfully")

CREATING DATA COLLATOR
✓ Data Collator Created Successfully


In [13]:
# ============================================================
# PART 7 : COMPUTE METRICS (WER & CER)
# ============================================================

print("=" * 70)
print("LOADING EVALUATION METRICS")
print("=" * 70)

import numpy as np
import evaluate

# ------------------------------------------------------------
# LOAD METRICS
# ------------------------------------------------------------

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print("✓ WER Metric Loaded")
print("✓ CER Metric Loaded")

# ------------------------------------------------------------
# COMPUTE METRICS FUNCTION
# ------------------------------------------------------------

def compute_metrics(pred):

    pred_logits = pred.predictions

    pred_ids = np.argmax(pred_logits, axis=-1)

    # Replace ignored labels
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions
    pred_str = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    # Decode labels
    label_str = processor.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    # Calculate metrics
    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": round(float(wer), 4),
        "cer": round(float(cer), 4),
    }

print("\n✓ Metrics Function Ready")
print("✓ Training pipeline is now ready.")

LOADING EVALUATION METRICS
✓ WER Metric Loaded
✓ CER Metric Loaded

✓ Metrics Function Ready
✓ Training pipeline is now ready.


In [14]:
# ============================================================
# PART 8 : TRAINING ARGUMENTS
# Compatible with Transformers 5.x
# ============================================================

print("=" * 70)
print("CREATING TRAINING ARGUMENTS")
print("=" * 70)

from transformers import TrainingArguments

training_args = TrainingArguments(

    # Output
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Mixed Precision
    fp16=torch.cuda.is_available(),

    # Data
    remove_unused_columns=False,
    dataloader_num_workers=2,

    # Logging
    report_to="none",

    # Seed
    seed=SEED,
)

print("✓ TrainingArguments Created Successfully")

CREATING TRAINING ARGUMENTS
✓ TrainingArguments Created Successfully


In [15]:
# ============================================================
# PART 9 : CREATE TRAINER
# Compatible with Transformers 5.13.1
# ============================================================

print("=" * 70)
print("CREATING TRAINER")
print("=" * 70)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✓ Trainer Created Successfully")

print("\n" + "=" * 70)
print("TRAINER SUMMARY")
print("=" * 70)

print(f"Training Samples   : {len(train_dataset)}")
print(f"Validation Samples : {len(validation_dataset)}")
print(f"Batch Size         : {BATCH_SIZE}")
print(f"Epochs             : {NUM_EPOCHS}")

print("\n✓ Ready for Training")

CREATING TRAINER
✓ Trainer Created Successfully

TRAINER SUMMARY
Training Samples   : 2103
Validation Samples : 295
Batch Size         : 8
Epochs             : 20

✓ Ready for Training


In [16]:
# ============================================================
# PART 10 : START TRAINING + SAVE MODEL
# Arabic ASR - FLEURS
# Compatible with Transformers 5.13.1
# ============================================================

print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)

import time
import json
import os

start_time = time.time()

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

train_result = trainer.train()

training_time = time.time() - start_time


# ------------------------------------------------------------
# TRAINING TIME
# ------------------------------------------------------------

hours = int(training_time // 3600)
minutes = int((training_time % 3600) // 60)
seconds = int(training_time % 60)


print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(
    f"Training Time : {hours:02d}:{minutes:02d}:{seconds:02d}"
)


# ------------------------------------------------------------
# TRAINING METRICS
# ------------------------------------------------------------

print("\nFinal Training Metrics")
print("-" * 70)

for key, value in train_result.metrics.items():
    print(f"{key}: {value}")


# ------------------------------------------------------------
# SAVE MODEL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAVING FINAL MODEL")
print("=" * 70)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# Save model weights + config
trainer.save_model(
    OUTPUT_DIR
)

# Save processor (feature extractor + tokenizer)
processor.save_pretrained(
    OUTPUT_DIR
)


# ------------------------------------------------------------
# SAVE TRAINING METRICS
# ------------------------------------------------------------

metrics_path = os.path.join(
    OUTPUT_DIR,
    "training_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(
        train_result.metrics,
        f,
        indent=4
    )


print("\n✓ Model Saved Successfully")
print(f"Saved Location : {OUTPUT_DIR}")

print("\nSaved Files:")
for file in os.listdir(OUTPUT_DIR):
    print(" -", file)

STARTING TRAINING


Epoch,Training Loss,Validation Loss,Wer,Cer
1,No log,1.114421,0.448000,0.134100
2,4.333436,0.857678,0.392600,0.114900
3,4.333436,0.744179,0.355300,0.099400
4,2.547337,0.697251,0.342700,0.091600
5,2.087611,0.692267,0.318900,0.084100
6,2.087611,0.630956,0.310200,0.079900
7,1.791737,0.623915,0.302400,0.078300
8,1.420668,0.648096,0.297800,0.077200
9,1.420668,0.652098,0.296600,0.076500
10,1.259471,0.699439,0.287300,0.075300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


TRAINING COMPLETED
Training Time : 06:29:30

Final Training Metrics
----------------------------------------------------------------------
train_runtime: 23369.9838
train_samples_per_second: 1.8
train_steps_per_second: 0.056
total_flos: 2.335918461619172e+19
train_loss: 1.4540639516079064
epoch: 20.0

SAVING FINAL MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Model Saved Successfully
Saved Location : ./wav2vec2-arabic-fleurs

Saved Files:
 - training_args.bin
 - config.json
 - training_metrics.json
 - processor_config.json
 - checkpoint-1320
 - vocab.json
 - tokenizer_config.json
 - checkpoint-1254
 - model.safetensors


In [ ]:
# ============================================================
# PART 11 : FINAL EVALUATION
# Arabic ASR - FLEURS
# Compatible with Transformers 5.13.1
# ============================================================

print("=" * 70)
print("EVALUATING FINAL MODEL")
print("=" * 70)

# ------------------------------------------------------------
# EVALUATE ON TEST DATASET
# ------------------------------------------------------------

test_results = trainer.evaluate(
    eval_dataset=test_dataset
)

print("\n" + "=" * 70)
print("TEST RESULTS")
print("=" * 70)

for key, value in test_results.items():
    print(f"{key}: {value}")


# ------------------------------------------------------------
# SAVE TEST RESULTS
# ------------------------------------------------------------

import json
import os

evaluation_path = os.path.join(
    OUTPUT_DIR,
    "test_results.json"
)

with open(evaluation_path, "w") as f:
    json.dump(
        test_results,
        f,
        indent=4
    )

print("\n✓ Evaluation Results Saved")
print(f"Location: {evaluation_path}")

In [ ]:
# ============================================================
# PART 12 : LOAD SAVED MODEL
# Arabic ASR - FLEURS
# Compatible with Transformers 5.13.1
# ============================================================

print("=" * 70)
print("LOADING SAVED ARABIC ASR MODEL")
print("=" * 70)

from transformers import AutoProcessor, AutoModelForCTC

# ------------------------------------------------------------
# MODEL PATH
# ------------------------------------------------------------

MODEL_PATH = OUTPUT_DIR


# ------------------------------------------------------------
# LOAD PROCESSOR
# ------------------------------------------------------------

inference_processor = AutoProcessor.from_pretrained(
    MODEL_PATH
)

print("✓ Processor Loaded")


# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

inference_model = AutoModelForCTC.from_pretrained(
    MODEL_PATH
)

inference_model.to(device)

inference_model.eval()

print("✓ Model Loaded")


# ------------------------------------------------------------
# MODEL INFO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERENCE MODEL READY")
print("=" * 70)

print("Model Path :", MODEL_PATH)
print("Device     :", device)
print("Vocabulary :", inference_model.config.vocab_size)

print("\n✓ Ready for Arabic Speech Recognition")

In [ ]:
# ============================================================
# PART 13 : ARABIC AUDIO INFERENCE
# Arabic ASR - FLEURS
# Compatible with Transformers 5.13.1
# ============================================================

print("=" * 70)
print("CREATING INFERENCE FUNCTION")
print("=" * 70)

import torch
import librosa
import numpy as np


def transcribe_audio(audio_path):

    # --------------------------------------------------------
    # LOAD AUDIO
    # --------------------------------------------------------

    speech, sr = librosa.load(
        audio_path,
        sr=16000
    )

    speech = speech.astype(np.float32)


    # --------------------------------------------------------
    # PROCESS AUDIO
    # --------------------------------------------------------

    inputs = inference_processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    )


    input_values = inputs.input_values.to(device)


    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    with torch.no_grad():

        logits = inference_model(
            input_values
        ).logits


    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )


    # --------------------------------------------------------
    # DECODE TEXT
    # --------------------------------------------------------

    transcription = inference_processor.batch_decode(
        predicted_ids
    )[0]


    return transcription



print("✓ Inference Function Created Successfully")


In [ ]:
# ============================================================
# PART 14 : TEST SAMPLE PREDICTIONS
# Arabic ASR - FLEURS
# ============================================================

print("=" * 70)
print("TESTING MODEL PREDICTIONS")
print("=" * 70)

import random
import torch
import numpy as np


# ------------------------------------------------------------
# SELECT RANDOM TEST SAMPLES
# ------------------------------------------------------------

num_samples = 5

sample_indices = random.sample(
    range(len(test_dataset)),
    num_samples
)


# ------------------------------------------------------------
# RUN INFERENCE
# ------------------------------------------------------------

for i, idx in enumerate(sample_indices):

    sample = test_dataset[idx]

    input_values = torch.tensor(
        sample["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)


    with torch.no_grad():

        logits = inference_model(
            input_values
        ).logits


    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )


    prediction = inference_processor.batch_decode(
        predicted_ids
    )[0]


    # Ground truth
    labels = sample["labels"]

    labels = [
        x for x in labels
        if x != -100
    ]

    reference = inference_processor.decode(
        labels,
        skip_special_tokens=True
    )


    print("\n" + "=" * 70)
    print(f"Sample {i+1}")
    print("-" * 70)

    print("REFERENCE:")
    print(reference)

    print("\nPREDICTION:")
    print(prediction)

print("\n✓ Testing Completed")